<a href="https://colab.research.google.com/github/sanidhyana742-web/AI-Placement-Prediction-System/blob/main/AI_Brain_Tumor_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving brain-tumor-mri-dataset.zip to brain-tumor-mri-dataset.zip


In [ ]:
!apt-get install unrar

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:
import zipfile

zip_ref = zipfile.ZipFile

zip_ref = zipfile.ZipFile("brain-tumor-mri-dataset.zip", 'r')
zip_ref.extractall("/content")
zip_ref.close()


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_path = "/content/Training"
test_path = "/content/Testing"

train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_dataset = train_datagen.flow_from_directory(
    train_path,
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical'
)

test_dataset = test_datagen.flow_from_directory(
    test_path,
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical'
)

Found 5712 images belonging to 4 classes.
Found 1311 images belonging to 4 classes.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

model = Sequential()

model.add(Conv2D(32,(3,3),activation='relu',input_shape=(128,128,3)))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(64,(3,3),activation='relu'))
model.add(MaxPooling2D(2,2))

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dense(4,activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
optimizer='adam',
loss='categorical_crossentropy',
metrics=['accuracy']
)

In [ ]:
model.fit(
train_dataset,
validation_data=test_dataset,
epochs=10
)

Epoch 1/10
179/179 ━━━━━━━━━━━━━━━━━━━━ 176s 976ms/step - accuracy: 0.7602 - loss: 0.6105 - val_accuracy: 0.8452 - val_loss: 0.4051
Epoch 2/10
179/179 ━━━━━━━━━━━━━━━━━━━━ 168s 937ms/step - accuracy: 0.9102 - loss: 0.2457 - val_accuracy: 0.9108 - val_loss: 0.2911
Epoch 3/10
179/179 ━━━━━━━━━━━━━━━━━━━━ 167s 931ms/step - accuracy: 0.9517 - loss: 0.1382 - val_accuracy: 0.9321 - val_loss: 0.1845
Epoch 4/10
179/179 ━━━━━━━━━━━━━━━━━━━━ 167s 931ms/step - accuracy: 0.9737 - loss: 0.0787 - val_accuracy: 0.9100 - val_loss: 0.2606
Epoch 5/10
179/179 ━━━━━━━━━━━━━━━━━━━━ 174s 967ms/step - accuracy: 0.9846 - loss: 0.0500 - val_accuracy: 0.9504 - val_loss: 0.1495
Epoch 6/10
179/179 ━━━━━━━━━━━━━━━━━━━━ 167s 933ms/step - accuracy: 0.9923 - loss: 0.0266 - val_accuracy: 0.9352 - val_loss: 0.2064
Epoch 7/10
179/179 ━━━━━━━━━━━━━━━━━━━━ 167s 928ms/step - accuracy: 0.9869 - loss: 0.0360 - val_accuracy: 0.9451 - val_loss: 0.2162
Epoch 8/10
179/179 ━━━━━━━━━━━━━━━━━━━━ 174s 971ms/step - accuracy: 0.9909 -

In [ ]:
model.save("brain_tumor_model.h5")

In [ ]:
from google.colab import files
files.download("brain_tumor_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
!pip install gradio opencv-python

In [25]:
from google.colab import files
uploaded = files.upload()

Saving brain_tumor_model.h5 to brain_tumor_model (1).h5


In [60]:
import gradio as gr
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
from tensorflow.keras.models import load_model

# Load trained CNN model
model = load_model("brain_tumor_model.h5")

classes = ["Glioma Tumor","Meningioma Tumor","Pituitary Tumor","No Tumor"]

def highlight_tumor(image):

    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # create heatmap
    heatmap = cv2.applyColorMap(gray, cv2.COLORMAP_JET)

    heat_gray = cv2.cvtColor(heatmap, cv2.COLOR_BGR2GRAY)

    _, thresh = cv2.threshold(heat_gray,200,255,cv2.THRESH_BINARY)

    contours,_ = cv2.findContours(thresh,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)

    result = image.copy()

    for cnt in contours:

        x,y,w,h = cv2.boundingRect(cnt)

        if w*h > 500:
            cv2.rectangle(result,(x,y),(x+w,y+h),(255,0,0),3)

    return result


def predict(image):

    time.sleep(2)   # AI analyzing animation

    img = cv2.resize(image,(128,128))
    img = img/255.0
    img = np.expand_dims(img, axis=0)

    prediction = model.predict(img)[0]

    class_index = np.argmax(prediction)
    confidence = prediction[class_index]

    result_text = f"{classes[class_index]} ({confidence*100:.2f}%)"

    highlighted = highlight_tumor(image)

    fig, ax = plt.subplots()
    ax.bar(classes, prediction)
    ax.set_title("Tumor Probability")
    ax.set_ylabel("Probability")

    return highlighted, result_text, fig


with gr.Blocks(theme=gr.themes.Soft(), title="AI Brain Tumor Detection") as demo:

    gr.Markdown("""
<style>

.banner{
width:100%;
padding:60px;
text-align:center;
border-radius:15px;
background: linear-gradient(135deg,#1f4cff,#00c6ff);
color:white;
margin-bottom:40px;
}

.banner h1{
font-size:48px;
font-weight:bold;
text-shadow:0 0 15px rgba(255,255,255,0.8),
             0 0 30px rgba(255,255,255,0.5);
}

.card-container{
display:flex;
gap:25px;
justify-content:center;
margin-top:30px;
}

.card{
background:white;
padding:25px;
border-radius:15px;
width:250px;
text-align:center;
box-shadow:0 8px 20px rgba(0,0,0,0.1);
transition:transform 0.3s, box-shadow 0.3s;
}

.card:hover{
transform:translateY(-10px);
box-shadow:0 15px 35px rgba(0,0,0,0.2);
}

.icon{
font-size:40px;
margin-bottom:10px;
}

</style>
""")

    # HOME PAGE

    with gr.Tab("🏠 Home"):

     gr.Markdown("""
<style>

.banner{
width:100%;
padding:70px;
text-align:center;
border-radius:15px;
background: linear-gradient(135deg,#1f4cff,#00c6ff);
color:white;
margin-bottom:40px;
}

.banner h1{
font-size:50px;
font-weight:bold;
text-shadow:0 0 15px rgba(255,255,255,0.9);
}

.banner p{
font-size:20px;
margin-top:10px;
}

.card-container{
display:flex;
justify-content:center;
gap:35px;
margin-top:40px;
flex-wrap:wrap;
}

.card{
background:white;
padding:30px;
border-radius:15px;
width:260px;
text-align:center;
box-shadow:0 10px 25px rgba(0,0,0,0.15);
transition:all 0.3s ease;
}

.card:hover{
transform:translateY(-12px);
box-shadow:0 20px 40px rgba(0,0,0,0.25);
}

.icon{
font-size:45px;
margin-bottom:10px;
}

.center-img{
display:flex;
justify-content:center;
margin-top:40px;
}

</style>

<div class="banner">
<h1>🧠 AI Brain Tumor Detection</h1>
<p>Advanced MRI Analysis Using Deep Learning</p>
</div>

<div class="card-container">

<div class="card">
<div class="icon">🧠</div>
<h3>Tumor Detection</h3>
<p>Detect brain tumors automatically from MRI scans.</p>
</div>

<div class="card">
<div class="icon">📊</div>
<h3>Probability Analysis</h3>
<p>AI model shows probability for each tumor type.</p>
</div>

<div class="card">
<div class="icon">⚡</div>
<h3>Fast AI Prediction</h3>
<p>Deep learning gives instant diagnosis results.</p>
</div>

</div>

<div class="center-img">
<img src="https://upload.wikimedia.org/wikipedia/commons/3/35/MRI_brain_sagittal_section.jpg" width="420">
</div>

""")




    # UPLOAD PAGE

    with gr.Tab("📤 Upload MRI"):

        input_image = gr.Image(
            sources=["upload","webcam"],
            type="numpy",
            label="Upload or Capture MRI Image"
        )

        detect_button = gr.Button("🧠 Run AI Detection")


    # RESULT PAGE

    with gr.Tab("🔬 Detection Results"):

        gr.Markdown("### 🧠 AI analyzing MRI scan...")

        with gr.Row():

            with gr.Column():
                with gr.Group():
                    gr.Markdown("### 🖼 MRI with Tumor Highlight")
                    output_image = gr.Image()

            with gr.Column():
                with gr.Group():
                    gr.Markdown("### 🧠 Prediction")
                    prediction_text = gr.Textbox()

            with gr.Column():
                with gr.Group():
                    gr.Markdown("### 📊 Tumor Probability")
                    probability_plot = gr.Plot()

        detect_button.click(
            predict,
            inputs=input_image,
            outputs=[output_image, prediction_text, probability_plot]
        )


demo.launch(share=True)

/tmp/ipykernel_419/3294670339.py:63: DeprecationWarning:

The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://de4003f533f39ab1bd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
